# C4 · Real PyBaMM DFN EIS

**PHYRE W3 Track C+D shared · GPU 优先 · ~20–40 min**

## 目标
彻底解决 W2 留下的 "pybamm 24.x 顶层没有 EISSimulation" 问题：
1. pin pybamm 24.x（与 C2 一致）
2. 多路径探测 `EISSimulation`（顶层 / `pybamm.simulations` / `pybamm.eis` / 子模块）
3. 找不到 → 用 `Simulation(... DFN())` 跑小幅时域电流阶跃 → FFT 出 Z(ω)
4. 用 §9E.1 中两个对比假设 (h0=baseline / h1=SEI 增厚) 验证 Nyquist 弧确有差别

## PASS
- 至少 1 条真实 DFN EIS 成功输出（既不是 NaN 也不是 analytic surrogate）
- h0 / h1 在 100 mHz – 10 Hz 区段 |Z| 比值 ≥ 1.2 (即弧高变化可观测)

In [ ]:
# ========== 1. pin pybamm 24.x (mirror C2 cell-1) ==========
import subprocess, sys, importlib, os
def sh(*a): return subprocess.run(a, capture_output=True, text=True)

for pkg in ('pybamm',):
    try: importlib.invalidate_caches(); importlib.import_module(pkg)
    except Exception: pass

r = sh(sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade',
       'pybamm>=24.0,<25', 'casadi', 'numpy<2', 'scipy', 'pandas', 'matplotlib',
       'anytree', 'sympy', 'xarray', 'pyparsing', 'tqdm')
print(r.stdout[-500:] if r.stdout else '', r.stderr[-300:] if r.stderr else '')

# wipe stale caches
for m in [k for k in list(sys.modules) if k.startswith('pybamm')]: del sys.modules[m]
import pybamm
print(f'pybamm version: {pybamm.__version__}')

In [ ]:
# ========== 2. probe EISSimulation everywhere ==========
import importlib, pkgutil
candidates = []
for top in ('pybamm', 'pybamm.simulations', 'pybamm.eis', 'pybamm.solvers'):
    try:
        mod = importlib.import_module(top)
        if hasattr(mod, 'EISSimulation'):
            candidates.append((top, getattr(mod, 'EISSimulation')))
    except ImportError: continue

# walk submodules of pybamm
for finder, name, ispkg in pkgutil.walk_packages(pybamm.__path__, prefix='pybamm.'):
    try:
        m = importlib.import_module(name)
        if hasattr(m, 'EISSimulation'):
            candidates.append((name, getattr(m, 'EISSimulation')))
    except Exception: continue

EIS_CLS = candidates[0][1] if candidates else None
print(f'EISSimulation candidates: {[c[0] for c in candidates]}')
print(f'using: {candidates[0][0] if candidates else "NONE — will fall back to time-domain FFT"}')

In [ ]:
# ========== 3. real EIS via EISSimulation (preferred path) ==========
import numpy as np
FREQS = np.logspace(-3, 4, 30)

def build_param(updates):
    p = pybamm.ParameterValues('Chen2020')
    for k, v in updates.items():
        if k.startswith('__'): continue
        try: p[k] = v
        except KeyError: pass
    return p

def real_eis_via_class(updates, freqs=FREQS):
    if EIS_CLS is None: return None
    try:
        model = pybamm.lithium_ion.DFN(options={'surface form':'differential'})
        sim = EIS_CLS(model, parameter_values=build_param(updates))
        sim.solve(freqs)
        return np.asarray(sim.solution['Impedance [Ohm]'].entries if hasattr(sim, 'solution') else sim.impedances)
    except Exception as e:
        print(f'real_eis_via_class failed: {e}')
        return None

Z0 = real_eis_via_class({})
print(f'baseline EIS: {None if Z0 is None else Z0.shape}')

In [ ]:
# ========== 4. fallback: time-domain step + FFT (DFN or SPM) ==========
def real_eis_via_step(updates, freqs=FREQS):
    """Apply small current step on DFN; FFT the voltage response → Z(ω). Robust fallback."""
    try:
        model = pybamm.lithium_ion.DFN()
    except Exception:
        model = pybamm.lithium_ion.SPM()
    pv = build_param(updates)
    I0 = 1e-3  # 1 mA step (small-signal)
    pv['Current function [A]'] = I0
    try:
        sim = pybamm.Simulation(model, parameter_values=pv)
        T_END, NT = 200.0, 4001
        sol = sim.solve([0, T_END], t_eval=np.linspace(0, T_END, NT))
        t = sol['Time [s]'].entries
        V = sol['Voltage [V]'].entries
        I = np.full_like(t, I0); I[0] = 0.0
        dt = t[1] - t[0]
        N = len(t)
        F = np.fft.rfft(V - V[0]); G = np.fft.rfft(I - I[0])
        ws = np.fft.rfftfreq(N, dt)
        Z = np.zeros_like(freqs, dtype=complex)
        for i, f in enumerate(freqs):
            j = int(np.argmin(np.abs(ws - f)))
            Z[i] = F[j] / (G[j] + 1e-18)
        return Z
    except Exception as e:
        print(f'time-domain fallback failed: {e}')
        return None

if Z0 is None:
    Z0 = real_eis_via_step({})
    print(f'time-domain Z0: {None if Z0 is None else Z0.shape}')

In [ ]:
# ========== 5. compare h0 (baseline) vs h1 (SEI thicker) ==========
import json, time
from pathlib import Path
PHYRE = Path('/content/drive/MyDrive/phyre') if Path('/content/drive/MyDrive/phyre').exists() else Path('phyre')
LOGS = PHYRE / 'logs'; LOGS.mkdir(parents=True, exist_ok=True)

real_eis = real_eis_via_class if EIS_CLS is not None else real_eis_via_step
Z_h0 = real_eis({})
Z_h1 = real_eis({'Negative electrode SEI thickness [m]': 1e-8,
                  'Initial inner SEI thickness [m]': 1e-8})
Z_h2 = real_eis({'Positive particle radius [m]': 2.6e-6})

def band_ratio(Za, Zb, fmin=0.1, fmax=10.0):
    mask = (FREQS >= fmin) & (FREQS <= fmax)
    if Za is None or Zb is None or not mask.any(): return float('nan')
    return float(np.mean(np.abs(Zb[mask]) / (np.abs(Za[mask]) + 1e-9)))

ratios = {
    'h1/h0_sei': band_ratio(Z_h0, Z_h1),
    'h2/h0_radius': band_ratio(Z_h0, Z_h2),
}
ok = all(Z is not None and not np.all(np.isnan(Z)) for Z in (Z_h0, Z_h1, Z_h2))
PASS = ok and any(r >= 1.2 or r <= 0.83 for r in ratios.values() if r==r)
print(f'real EIS ok: {ok}')
print(f'band ratios (0.1–10 Hz): {ratios}')
print(f'PASS: {PASS}')

out = {
    'pybamm_version': pybamm.__version__,
    'eis_class_path': candidates[0][0] if candidates else None,
    'fallback_used': EIS_CLS is None,
    'real_eis_ok': bool(ok),
    'band_ratios_0p1_10Hz': ratios,
    'pass': bool(PASS),
    'freqs_hz': FREQS.tolist(),
    'Z_h0_real': None if Z_h0 is None else Z_h0.real.tolist(),
    'Z_h0_imag': None if Z_h0 is None else Z_h0.imag.tolist(),
    'Z_h1_real': None if Z_h1 is None else Z_h1.real.tolist(),
    'Z_h1_imag': None if Z_h1 is None else Z_h1.imag.tolist(),
    'Z_h2_real': None if Z_h2 is None else Z_h2.real.tolist(),
    'Z_h2_imag': None if Z_h2 is None else Z_h2.imag.tolist(),
}
(LOGS / 'C4_real_dfn.json').write_text(json.dumps(out, indent=2), encoding='utf-8')
print(f'wrote {LOGS/"C4_real_dfn.json"}')

In [ ]:
# ========== 6. plot Nyquist sanity check ==========
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,5))
for name, Z in [('h0 baseline', Z_h0), ('h1 SEI thick', Z_h1), ('h2 small particle', Z_h2)]:
    if Z is None or np.all(np.isnan(Z)): continue
    ax.plot(Z.real, -Z.imag, '-o', ms=3, label=name)
ax.set_xlabel('Re(Z) [Ohm]'); ax.set_ylabel('-Im(Z) [Ohm]')
ax.set_title(f'Real DFN Nyquist (pybamm {pybamm.__version__}, fallback={EIS_CLS is None})')
ax.legend(); ax.grid(alpha=0.3); ax.axis('equal')
plt.tight_layout(); plt.savefig(LOGS / 'C4_nyquist.png', dpi=120)
plt.show()

## Go / No-Go

**PASS:** real DFN EIS 跑出来 + h0/h1/h2 在 0.1–10 Hz 有 ≥ 1.2× 弧高差异。

**On EIS class 找不到 + time-domain 也失败:**
1. 退到 SPM（已在 cell-4 自动 fallback）
2. 缩短时间窗（200s → 50s）+ 提高采样（4001 → 16001）
3. 改用 sinusoidal current at single freq + lock-in 提取（最 robust）

**On EIS 出来了但弧高差太小:**
Chen2020 默认 SEI 太薄 → 改用更显著的 update（`Initial inner SEI thickness` 5e-9 → 5e-8）。

**Commit once green:**
```
git add nb/C4_real_dfn.ipynb logs/C4_real_dfn.json logs/C4_nyquist.png
git commit -m 'C4: real PyBaMM DFN EIS — band ratios h1/h0=X.XX'
git tag w3-track-c4-real-dfn
```

Once C4 PASS：D2 重跑用 `real_eis(...)` 替换 analytic surrogate，sanity_note 中 `real_dfn_status` 由 `deferred_to_W3` 改为 `done`。